# Phase 1 — Data Preparation
## Goal: Load ViSEC, verify structure, split dataset, save split indices

In [ ]:
# Install dependencies (run once)
# !pip install datasets transformers torch torchaudio \
#              librosa scikit-learn matplotlib seaborn soundfile tqdm

In [ ]:
import sys
sys.path.append("../src")
from config import *

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Audio
from sklearn.model_selection import train_test_split
from collections import Counter
import random, torch

# Fix seed globally
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Seed: {SEED}")

## Step 1 — Load dataset

In [ ]:
ds = load_dataset(DATASET_NAME, split="train")
ds = ds.cast_column("path", Audio(sampling_rate=16000))
print(f"Total samples : {len(ds)}")
print(f"Columns       : {ds.column_names}")
print(f"\nSample[0]:")
sample = ds[0]
for k, v in sample.items():
    if k == "path":
        print(f"  audio: array shape={v['array'].shape}, sr={v['sampling_rate']}")
    else:
        print(f"  {k}: {v}")

## Step 2 — Verify labels

In [ ]:
emotions = ds["emotion"]
unique_emotions = sorted(set(emotions))
label_counts = Counter(emotions)

print("Unique emotions:", unique_emotions)
print("\nLabel distribution:")
for emo, count in sorted(label_counts.items()):
    print(f"  {emo:10s}: {count:4d} samples")

# Check if labels match LABEL_MAP in config
missing = [e for e in unique_emotions if e not in LABEL_MAP]
if missing:
    print(f"\n\u26a0\ufe0f  WARNING: Labels not in LABEL_MAP: {missing}")
else:
    print(f"\n\u2705 All labels match config LABEL_MAP")

## Step 3 — Stratified split 70/15/15

In [ ]:
all_indices = list(range(len(ds)))
all_labels  = [LABEL_MAP[e] for e in ds["emotion"]]

# Split 1: separate 70% train
train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    stratify=all_labels,
    random_state=SEED
)

# Split 2: split remainder into 15% val + 15% test
temp_labels = [all_labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED
)

print(f"Train : {len(train_idx):4d} samples ({len(train_idx)/len(ds)*100:.1f}%)")
print(f"Val   : {len(val_idx):4d} samples ({len(val_idx)/len(ds)*100:.1f}%)")
print(f"Test  : {len(test_idx):4d} samples ({len(test_idx)/len(ds)*100:.1f}%)")
print(f"Total : {len(train_idx)+len(val_idx)+len(test_idx):4d} samples")

## Step 4 — Verify label distribution per split

In [ ]:
def count_labels(indices):
    labels = [all_labels[i] for i in indices]
    return Counter(labels)

splits_counts = {
    "Train": count_labels(train_idx),
    "Val"  : count_labels(val_idx),
    "Test" : count_labels(test_idx),
}

# Pretty-print table
df_dist = pd.DataFrame(splits_counts).T
df_dist.columns = LABEL_NAMES
df_dist["Total"] = df_dist.sum(axis=1)
print(df_dist.to_string())

## Step 5 — Plot label distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4CAF50", "#2196F3", "#FF9800", "#F44336"]

for ax, (split_name, idx) in zip(axes, [
    ("Train", train_idx), ("Val", val_idx), ("Test", test_idx)
]):
    counts = count_labels(idx)
    bars = ax.bar(
        LABEL_NAMES,
        [counts[v] for v in range(NUM_CLASSES)],
        color=colors
    )
    ax.set_title(f"{split_name} ({len(idx)} samples)", fontsize=13)
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    # Annotate bar heights
    for bar, val in zip(bars, [counts[v] for v in range(NUM_CLASSES)]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha="center", fontsize=10)

plt.suptitle("ViSEC Label Distribution per Split", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/phase1_label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: phase1_label_distribution.png")

## Step 6 — Verify audio format

In [ ]:
# Check 5 random samples
check_indices = random.sample(all_indices, 5)
print("Checking audio format on 5 random samples:\n")
sr_set = set()
for i in check_indices:
    audio = ds[i]["path"]
    sr    = audio["sampling_rate"]
    dur   = len(audio["array"]) / sr
    sr_set.add(sr)
    print(f"  Sample {i:4d} | sr={sr}Hz | duration={dur:.2f}s | "
          f"emotion={ds[i]['emotion']}")

if sr_set == {16000}:
    print(f"\n\u2705 All checked samples are 16kHz \u2014 no resampling needed")
else:
    print(f"\n\u26a0\ufe0f  Found sampling rates: {sr_set} \u2014 resampling required in Phase 2")

## Step 7 — Duration statistics

In [ ]:
durations = np.array(ds["duration"])
print("Duration statistics (seconds):")
print(f"  Mean   : {durations.mean():.2f}s")
print(f"  Std    : {durations.std():.2f}s")
print(f"  Min    : {durations.min():.2f}s")
print(f"  Max    : {durations.max():.2f}s")
print(f"  Median : {np.median(durations):.2f}s")

# Histogram
plt.figure(figsize=(8, 4))
plt.hist(durations, bins=40, color="#2196F3", edgecolor="white")
plt.axvline(durations.mean(), color="red", linestyle="--", label=f"Mean={durations.mean():.2f}s")
plt.title("ViSEC Audio Duration Distribution")
plt.xlabel("Duration (seconds)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/phase1_duration_distribution.png", dpi=150)
plt.show()
print("Saved: phase1_duration_distribution.png")

## Step 8 — Save splits (Phase 2 and 3 reuse this file)

In [ ]:
splits_data = {
    "train"     : train_idx,
    "val"       : val_idx,
    "test"      : test_idx,
    "label_map" : LABEL_MAP,
    "seed"      : SEED,
    "dataset"   : DATASET_NAME,
    "total"     : len(ds),
    "stats": {
        "train": len(train_idx),
        "val"  : len(val_idx),
        "test" : len(test_idx),
    }
}
save_path = f"{EMBED_DIR}/splits.json"
with open(save_path, "w") as f:
    json.dump(splits_data, f, indent=2)

print(f"\u2705 Saved splits.json \u2192 {save_path}")
print(f"\nContent preview:")
print(json.dumps({k: v for k, v in splits_data.items() if k != "train"
                  and k != "val" and k != "test"}, indent=2))

## \u2705 Phase 1 Complete — Checklist

In [ ]:
import os
checks = {
    "splits.json saved"              : os.path.exists(f"{EMBED_DIR}/splits.json"),
    "label_distribution.png saved"   : os.path.exists(f"{FIGURES_DIR}/phase1_label_distribution.png"),
    "duration_distribution.png saved" : os.path.exists(f"{FIGURES_DIR}/phase1_duration_distribution.png"),
    "All labels in LABEL_MAP"        : len(missing) == 0,
    "Audio is 16kHz"                 : sr_set == {16000},
}
print("=== Phase 1 Checklist ===")
for item, passed in checks.items():
    icon = "\u2705" if passed else "\u274c"
    print(f"  {icon} {item}")

all_passed = all(checks.values())
if all_passed:
    print("\n\U0001f389 Phase 1 PASSED \u2014 Ready for Phase 2")
else:
    print("\n\u26a0\ufe0f  Some items failed \u2014 check before proceeding")